### Multiclass classification

When we have more than two classes as an option, it's known as **multi-class classification**.
* This means if you have 3 different classes, it'a multiclass classification.
* It alos means if we have 1000 different classes, that is also multiclass classification.

To practice multi-class classification , we're going to build a neural network to classify images of different itemns of clothing.

*we will use **Fashion MNIST** dataset*

In [ ]:
import tensorflow as tf
from tensorflow.keras.datasets import fashion_mnist

# The data has already been sorted into training and test sets for us
(train_data, train_labels), (test_data, test_labels) = fashion_mnist.load_data()

In [ ]:
# Show the foirst training examples
print(f'Training sample:\n{train_data[0]}\n')
print(f'Training label:\n{train_labels[0]}\n')

In [ ]:
# check the shape of single examample
train_data[0].shape, train_labels[0].shape

In [ ]:
# plot a single sample
import matplotlib.pyplot as plt
plt.imshow(train_data[7])

In [ ]:
# check out the smaple labels
train_labels[7]

In [ ]:
# create a small list so we can index onto our training labels so they're human readable
class_names = ['T-shirt/top', 'Trouser', 'Pullover', 'Dress', 'Coat',
               'Sandal', 'Shirt', 'Sneaker', 'Bag', 'Ankle boot']

len(class_names)

In [ ]:
# plot an example image and its label
index_of_choice = 1232
plt.imshow(train_data[index_of_choice], cmap=plt.cm.binary)
plt.title(class_names[train_labels[index_of_choice]])

In [ ]:
# plot multiple random image of fashion MNIST
# plot 4 random items
import random
plt.figure(figsize=(7,7))
for i in range(4):
    az = plt.subplot(2, 2, i+1)
    rand_index = random.choice(range(len(train_data)))
    plt.imshow(train_data[rand_index], cmap=plt.cm.binary)
    plt.title(class_names[train_labels[rand_index]])
    plt.axis(False)

### let's build a multiclass classification model

For our multiclass classification model, we can use a similar architecture to our binary classifiers, however, we're going to tweak a few things:
* Input shape = (28, 28) the shape of one image
* Output shape = 10
* Loss function = tf.keras.losses.CategoricalCrossentropy()
  * If your labels are one-hot encoded, use CateegoricalCrossentropy()
  * If your labels are in integer form use SparseCateegoricalCrossentropy()
* Output activation = softmax (not sigmoid)

In [ ]:
flatten_model = tf.keras.Sequential([tf.keras.layers.Flatten(input_shape=(28,28))])
flatten_model.output_shape

In [ ]:
28 * 28

In [ ]:
# our data needs to be flattened from (28 * 28 to None, 784)
flattened_model = tf.keras.Sequential([tf.keras.layers.Flatten(input_shape=(28,28))])
flatten_model.output_shape

In [ ]:
tf.one_hot(train_labels[:10], depth=10)

In [ ]:
# set random seed
tf.random.set_seed(42)

# 1. create a model
model = tf.keras.Sequential([
    tf.keras.layers.Flatten(input_shape=(28,28)),
    tf.keras.layers.Dense(4, activation='relu'),
    tf.keras.layers.Dense(4, activation='relu'),
    tf.keras.layers.Dense(10, activation=tf.keras.activations.softmax)
])

# 2. compile the model
model.compile(loss=tf.keras.losses.CategoricalCrossentropy(),     # SparseCategoricalCrossentropy,
              optimizer=tf.keras.optimizers.Adam(),
              metrics=['accuracy'])

# 3. fit the model
non_norm_history = model.fit(train_data,
                             tf.one_hot(train_labels, depth=10),
                             epochs=10,
                             validation_data=(test_data, tf.one_hot(test_labels, depth=10)))

In [ ]:
# Check the model summary
model.summary()

In [ ]:
# check min and max of training data
train_data.min(), tf.math.reduce_max(train_data).numpy()

Neural networtk prefer dqata to be scaled (or normalized) this means they like to have numbers in the tensors they try to find patterns between 0 and 1.

In [ ]:
# we can get our training ans test data between 0 and 1 by dividing by the maximumn number
train_data_norm = train_data / 255.0
test_data_norm = test_data / 255.0

In [ ]:
# check the min and max of scaled data
tf.math.reduce_min(train_data_norm).numpy(), tf.math.reduce_max(test_data_norm).numpy()

In [ ]:
# using ecxact same model with normalized data
# set random seed
tf.random.set_seed(42)

# 1. create a model
norm_model = tf.keras.Sequential([
    tf.keras.layers.Flatten(input_shape=(28,28)),
    tf.keras.layers.Dense(4, activation='relu'),
    tf.keras.layers.Dense(4, activation='relu'),
    tf.keras.layers.Dense(10, activation=tf.keras.activations.softmax)
])

# 2. compile the model
norm_model.compile(loss=tf.keras.losses.CategoricalCrossentropy(),     # SparseCategoricalCrossentropy,
              optimizer=tf.keras.optimizers.Adam(),
              metrics=['accuracy'])

# 3. fit the model
norm_history = norm_model.fit(train_data_norm,
                             tf.one_hot(train_labels, depth=10),
                             epochs=10,
                             validation_data=(test_data_norm, tf.one_hot(test_labels, depth=10)))

🔑 **Note**: Neural Networks tend to ptrefer data in numerical form as well as scaled/normalized (numbers between 0 & 1).

### plotting the loss curves

In [ ]:
# loss curves of each model
import pandas as pd

# plot non-normalized data
pd.DataFrame(non_norm_history.history).plot(title='Non-Normalized data')
# plot normalized data
pd.DataFrame(norm_history.history).plot(title='Normalized data')

> 🔑 **Note**: The same model with even *slightly* differnet data can produce *dramatically* different results. so when you're comparing them on the same criteria (e.g. same architecture but different data or same data but different architecture).

### Findig the ideal learning rate

In [ ]:
# set random seed
tf.random.set_seed(42)

# create a model
model_1 = tf.keras.Sequential([
    tf.keras.layers.Flatten(input_shape=(28,28)),
    tf.keras.layers.Dense(4, activation='relu'),
    tf.keras.layers.Dense(4, activation='relu'),
    tf.keras.layers.Dense(10, activation=tf.keras.activations.softmax)
])

# compile the model
model_1.compile(loss=tf.keras.losses.CategoricalCrossentropy(),     # SparseCategoricalCrossentropy,
              optimizer=tf.keras.optimizers.Adam(),
              metrics=['accuracy'])

# create the learninf rate callback
lr_scheduler = tf.keras.callbacks.LearningRateScheduler(lambda epoch: 1e-3 * 10 ** (epoch/20))

# fit the model
find_lr_history = model_1.fit(
                            train_data_norm,
                            tf.one_hot(train_labels, depth=10), # Clean integer labels
                            epochs=40,
                            validation_data=(test_data_norm, tf.one_hot(test_labels, depth=10)), # Clean integer labels
                            callbacks=[lr_scheduler])

In [ ]:
# plot learninf rate decay curves
import matplotlib.pyplot as plt

lrs = 1e-3 * (10 ** (tf.range(40)/20))
plt.semilogx(lrs, find_lr_history.history['loss'])
plt.xlabel('Learning rate')
plt.ylabel('Loss')
plt.title('FInding the Ideal Learninf rate')
plt.show()

In [ ]:
# let refit the model with Ideal Learning rate

# set random seed
tf.random.set_seed(42)

# create a model
model_2 = tf.keras.Sequential([
    tf.keras.layers.Flatten(input_shape=(28,28)),
    tf.keras.layers.Dense(4, activation='relu'),
    tf.keras.layers.Dense(4, activation='relu'),
    tf.keras.layers.Dense(10, activation=tf.keras.activations.softmax)
])

# compile the model
model_2.compile(loss=tf.keras.losses.SparseCategoricalCrossentropy(),     # SparseCategoricalCrossentropy,
              optimizer=tf.keras.optimizers.Adam(),
              metrics=['accuracy'])

# create the learninf rate callback
lr_scheduler = tf.keras.callbacks.LearningRateScheduler(lambda epoch: 1e-3 * 10 ** (epoch/20))

# fit the model
find_lr_history = model_2.fit(
                            train_data_norm,
                            train_labels, # Clean integer labels
                            epochs=20,
                            validation_data=(test_data_norm, test_labels), # Clean integer labels
)

### Evaluating our multiclass classification model

To evaluate our multiclass classification model we could:
* Evaluate its performance using other classification metrics (such as confusion matrix).
*  Asees some of its predictions (through visualisation).
* Improve its results (by training for longer or changing its architecture).
* Save and export it for use in a application.

Let's go through the top 2...

In [ ]:
import itertools
import numpy as np

figsize = (10, 10)

# create the confusion matrix
def make_confusion_matrix(y_true, y_pred, classes=None, figsize=(10,10), text_size=15):
    cm = confusion_matrix(y_true, y_pred)
    cm_norm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]  # normalize our confusion matrix
    n_classes = cm.shape[0]

    # let's prettify it
    fig, ax = plt.subplots(figsize=figsize)
    # creat a matrix plot
    cax = ax.matshow(cm, cmap=plt.cm.Blues)
    fig.colorbar(cax)

    # set labels to be classes
    if classes:
        labels = classes
    else:
        labels = np.arange(cm.shape[0])

    # labe; the axes
    ax.set(title="Confusion Matrix",
            xlabel="Predicted Label",
            ylabel="True Label",
            xticks=np.arange(n_classes),
            yticks=np.arange(n_classes),
            xticklabels=labels,
            yticklabels=labels)

    # set x axis label to bottom
    ax.xaxis.set_label_position('bottom')
    ax.xaxis.tick_bottom()

    # Adjust label size
    ax.yaxis.label.set_size(text_size)
    ax.xaxis.label.set_size(text_size)
    ax.title.set_size(text_size)


    # set threshold for different colors
    threshold = (cm.max() + cm.min()) / 2.

    # plot the text on each cell
    for i, j in itertools.product(range(cm.shape[0]), range(cm.shape[1])):
        plt.text(j, i, f"{cm[i, j]}  ({cm_norm[i,j]*100:.1f}%)",
                horizontalalignment='center',
                color="white" if cm[i, j] > threshold else "black",
                size=15)

In [ ]:
class_names

In [ ]:
# make some predictions with our models
y_probs  = model_2.predict(test_data_norm)  # probs is short for predictions probabilities

# view the first 5 predictions 
y_probs[:5]

> 🔑 **Note** Remember to make predictions on the same kind of data your model was trained on (e.g. if your model wad trained on normalized data, you'll want to make predictions on normalized data).

In [ ]:
y_probs[0], tf.argmax(y_probs[0]), class_names[tf.argmax(y_probs[0])]

In [ ]:
## convert all of the predictions  probabilities into integer
y_preds = y_probs.argmax(axis=1)

# View the first 10 predictions labels
y_preds[:10]

In [ ]:
test_labels[:10]

In [ ]:
from sklearn.metrics._plot.confusion_matrix import confusion_matrix
confusion_matrix(y_true=test_labels,
                 y_pred=y_preds)

In [ ]:
# make a prttier confusion matrix
make_confusion_matrix(y_true=test_labels,
                      y_pred=y_preds,
                      classes=class_names,
                      figsize=(25, 20),
                      text_size=10)

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay

cm = confusion_matrix(y_true=test_labels,
                      y_pred=y_preds)


plt.rcParams.update({'font.size':14})
fig, ax = plt.subplots(figsize=(12, 12))
display = ConfusionMatrixDisplay(cm, 
                       display_labels=class_names)

display.plot(ax=ax,
             cmap='Blues', 
             xticks_rotation=45)
plt.title('COnfusion Matrix', fontsize=18)
plt.show()

🔑 **Note**: Often when working with images and other forms of visual data, it's good to visualise as much as possible to develop a further understanding of the data and the inputs and outputs of the model.

How about we create a fun litte function for:
* Plot a random image
* Make a prediction on said image
* Label the plot with the truth label and the prediction label

In [ ]:
# plot a random image
import random

def plot_random_image(model, images, true_labels, classes):
    '''
    Picks a random image, plot it and label it with a prediction and truth label.
    ''' 
    
    # set up a random integer
    i = random.randint(0, len(images))
    
    # create prediction and images
    target_image = images[i]
    pred_probs = model.predict(target_image.reshape(1, 28, 28))
    pred_label = classes[pred_probs.argmax()]
    true_label = classes[true_labels[i]]
    
    # plot the image
    plt.imshow(target_image, cmap=plt.cm.binary)
    
    # change the plotcolor of the titles depending on if the prediction is roght or wrong
    if pred_label == true_label:
        color = 'green'
    else:
        color = 'red'
        
    # add xlabel information (prediction/true label)
    plt.xlabel("Pred: {} {:2.0f}& (True: {})".format(pred_label,
                                                     100*tf.reduce_max(pred_probs),
                                                     true_label), 
               color=color) # set the color based on prediction

In [ ]:
# check out the random image and its prediction
plot_random_image(model=model_2,
                  images=test_data_norm,
                  true_labels=test_labels,
                  classes=class_names)

### What `patterns` is our model learning???

In [ ]:
# Find the layers of our model recent layers
model_2.layers

In [ ]:
# Extract a particular layer
model_2.layers[1]

In [ ]:
# Get the patterns of a layer in our network
weights, biases  = model.layers[1].get_weights()

# Shapes
weights, weights.shape

In [ ]:
model_2.summary()

Now let's check out the bias vector

In [ ]:
# Bias and bias shapes
biases, biases.shape

Every neuron has a bias vector, Each of these is paired with a weights matrix.

The Bias vector get initialized as zeros (at least in the case of a TensorFLow Dense layers).

The bias vector dictates how much the patterns within the corresponding weights matrix should influence the next layer.

In [ ]:
# let's check out another way of viewing our deep learning modeks
from tensorflow.keras.utils import plot_model

# see tge inputs and outputs of each layer
plot_model(model_2, show_shapes=True)

`plt.imshow()` test

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Create a 2D array (grayscale image)
img = np.random.rand(100, 100)

plt.imshow(img, cmap='gray')
plt.colorbar()  # Show color scale
plt.title("Grayscale Image")
plt.show()

In [ ]:
data = np.random.rand(10, 10)

plt.imshow(data, cmap='viridis')
plt.colorbar()
plt.title("Heatmap")
plt.show()